# Hemlock — Attack Walkthrough

End-to-end demonstration of a Direct Prompt Injection attack against a RAG pipeline.

**What you'll see:**
1. A clean pipeline answering correctly
2. A poisoned pipeline that follows injected instructions
3. The retrieval trace — which chunks were fetched and why
4. How defense layers block or miss the attack

No API key required — uses `MockLLM` and in-memory ChromaDB.

In [ ]:
# Install hemlock-rag if not already installed
# !pip install hemlock-rag

## 1. Setup — MockLLM + in-memory pipeline

In [ ]:
import sys
sys.path.insert(0, "..")

from tests.conftest import MockLLM  # zero-cost mock; echoes context
from hemlock.pipeline import Pipeline

def make_pipeline():
    return Pipeline(
        llm=MockLLM(),
        persist_dir=None,   # in-memory, no disk writes
        collection="lab_demo",
    )

pipeline = make_pipeline()
print("Pipeline ready:", pipeline)

## 2. Baseline — clean pipeline

In [ ]:
pipeline.reset()
pipeline.ingest_text("The capital of France is Paris.")
pipeline.ingest_text("The capital of Germany is Berlin.")
pipeline.ingest_text("The capital of Italy is Rome.")

trace = pipeline.query("What is the capital of France?")
print("Query:", trace.query)
print("Response:", trace.response)
print("Injected:", trace.injected)
print("\nChunks retrieved:")
for i, chunk in enumerate(trace.chunks, 1):
    print(f"  {i}. {chunk.page_content[:80]}")

## 3. Attack — Direct Prompt Injection (explicit variant)

In [ ]:
from attacks.direct_injection import DirectInjection

attack = DirectInjection(pipeline, variant="explicit")
result = attack.run()

print(f"Attack: {result.attack_name}")
print(f"Succeeded: {result.succeeded}")
print(f"Response: {result.trace.response}")
print(f"Notes: {result.notes}")

## 4. Retrieval trace — visualise what the model saw

In [ ]:
from IPython.display import HTML

rows = "".join(
    f"<tr><td>{i}</td><td style='font-family:monospace;font-size:12px'>{c.page_content[:200]}</td>"
    f"<td>{c.metadata.get('source','')}</td></tr>"
    for i, c in enumerate(result.trace.chunks, 1)
)

HTML(f"""
<table border='1' style='border-collapse:collapse;width:100%'>
  <tr><th>#</th><th>Chunk content</th><th>Source</th></tr>
  {rows}
</table>
""")

## 5. Run all 3 variants

In [ ]:
for variant in DirectInjection.VARIANTS:
    p = make_pipeline()
    atk = DirectInjection(p, variant=variant)
    r = atk.run()
    status = "✗ SUCCEEDED" if r.succeeded else "✓ blocked"
    print(f"  [{variant:16s}]  {status}")

## 6. Defense — InjectionPatternFilter

In [ ]:
from defenses.input_sanitizer import InjectionPatternFilter
from langchain_core.documents import Document

defense = InjectionPatternFilter()

for variant in DirectInjection.VARIANTS:
    p = make_pipeline()
    atk = DirectInjection(p, variant=variant)
    malicious = Document(page_content=atk._malicious_doc, metadata={"source": "malicious"})
    result_doc, report = defense.inspect(malicious)
    blocked = result_doc is None
    print(f"  [{variant:16s}]  {'BLOCKED' if blocked else 'passed'}  — {report.detail}")

## 7. Full scorer — all attacks × all hardening levels

In [ ]:
from attacks.registry import ATTACK_REGISTRY
from hemlock.scorer import Scorer, print_report

p = make_pipeline()
scorer = Scorer(
    pipeline=p,
    attacks=list(ATTACK_REGISTRY.values()),
    model_name="MockLLM",
)

report = scorer.run(verbose=True)
print_report(report)

In [ ]:
# Export the report as HTML
html = report.to_html()
with open("report.html", "w") as f:
    f.write(html)
print("Report saved to report.html")